# 02 Volume check: Part A (build years only)
Build years = 1999-01-04 to 2020-12-31. **Nothing dated 2021 or later is loaded or computed here** (the file is filtered while reading; every checkpoint's next-month outcome also ends by 2020-12-31).

Tests the dashboard's volume rule, copied exactly:
- **Past month change** = close today / close 21 trading days ago - 1 (adjusted close).
- **Up-day share** = over the last 21 daily changes, volume on days the close was higher than the day before, divided by all volume in those 21 days, times 100. A day with an unchanged close counts as not up. (Up/down uses adjusted close.)
- **Confirms** = (change > 0 and share >= 55) or (change < 0 and share <= 45). **Doesn't confirm** = (change > 0 and share <= 45) or (change < 0 and share >= 55). Otherwise **Neutral**. In plain words: "Confirms" = the move came with volume on the same side (a rise on mostly up-day volume, a fall on mostly down-day volume).
- **Checkpoints** every 21 trading days (non-overlapping). At each, a symbol is labeled if it has the full history and nonzero volume on at least 18 of the 21 days.
- **Outcome** = next 21 trading days' change in adjusted close. **Continued** = outcome has the same sign as the past month change.

Two views: **price** (the dashboard's rule; the main test) and **vs average sector** (each fund's change, daily moves and outcome minus the equal-weight average of the sector funds that exist on those days; second view).

In [1]:
import sys
from pathlib import Path
ROOT = Path.cwd().resolve()
if not (ROOT / "config.py").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
import numpy as np, pandas as pd
import config
from rotation.data import load_prices, wide
pd.set_option("display.width", 220, "display.max_columns", 40, "display.max_rows", 300)
R = config.RESULTS_DIR; R.mkdir(exist_ok=True)
N = 21                       # trading-day window
MIN_VOL_DAYS = 18            # nonzero-volume days needed out of 21
SYMS = [s for s in config.ALL_SYMBOLS if s != "BTC-USD"]     # 23 symbols (no volume rule for BTC)
GROUPS = {s: ("sector" if s in config.SECTORS else "industry" if s in config.INDUSTRIES else
              "other markets" if s in config.OTHER else "benchmark (SPY)") for s in SYMS}

## 1. Load build years only (adjusted close and volume)
Volume was saved by notebook 00 in the same Yahoo Finance download as prices (`volume` column of `prices.parquet`); nothing is re-downloaded.

In [2]:
df = load_prices(symbols=SYMS, fields=["adj_close", "volume"], end=config.BUILD_END)
assert df.date.max() <= pd.Timestamp(config.BUILD_END)
P = wide(df, "adj_close")[SYMS]
V = wide(df, "volume")[SYMS]
print("days:", len(P), "|", P.index.min().date(), "to", P.index.max().date(), "|", P.shape[1], "symbols")
cov = pd.DataFrame({"first_date": P.apply(lambda c: c.first_valid_index().date()),
                    "days": P.notna().sum(), "zero_or_missing_volume_days": (V.fillna(0) == 0).sum() - P.isna().sum()})
cov

days: 5536 | 1999-01-04 to 2020-12-31 | 23 symbols


,first_date,days,zero_or_missing_volume_days
symbol,,,
SPY,1999-01-04,5536,0
XLK,1999-01-04,5536,0
XLE,1999-01-04,5536,0
XLV,1999-01-04,5536,0
XLU,1999-01-04,5536,0
XLF,1999-01-04,5536,0
XLI,1999-01-04,5536,1
XLB,1999-01-04,5536,0
XLP,1999-01-04,5536,0


## 2. Build the rule at every day (rolling), then pick checkpoints
Both views use the same function; only the inputs differ (price view: own daily change; vs-sector view: own daily change minus the sector-average daily change).

In [3]:
V0 = V.fillna(0)
valid_hist = P.notna().astype(int).rolling(N + 1, min_periods=N + 1).sum() == N + 1     # prices on t-21..t all exist
nz_days = (V0 > 0).astype(int).rolling(N, min_periods=N).sum()
eligible = valid_hist & (nz_days >= MIN_VOL_DAYS)

S = P[config.SECTORS]
sec_change = (S / S.shift(N) - 1).mean(axis=1, skipna=True)            # equal-weight average of sector 21-day changes
sec_outcome = (S.shift(-N) / S - 1).mean(axis=1, skipna=True)          # same, next 21 days
sec_daily = S.pct_change(fill_method=None).mean(axis=1, skipna=True)   # equal-weight average daily change

def up_share(daily_move):
    up = (daily_move > 0)                                               # unchanged or missing = not up
    up_vol = V0.where(up, 0.0).rolling(N, min_periods=N).sum()
    tot = V0.rolling(N, min_periods=N).sum()
    return 100 * up_vol / tot.where(tot > 0)

d_price = P.diff()                                                      # sign of the daily change in adjusted close
d_rel = P.pct_change(fill_method=None).sub(sec_daily, axis=0)           # daily change minus sector-average daily change
change = {"price": P / P.shift(N) - 1, "vs sector": (P / P.shift(N) - 1).sub(sec_change, axis=0)}
outcome = {"price": P.shift(-N) / P - 1, "vs sector": (P.shift(-N) / P - 1).sub(sec_outcome, axis=0)}
share = {"price": up_share(d_price), "vs sector": up_share(d_rel)}

cp = np.arange(N, len(P), N)                 # checkpoint positions 21, 42, 63, ... (non-overlapping)
rows = []
for view in change:
    for i in cp:
        date = P.index[i]
        ch, sh, out, ok = change[view].iloc[i], share[view].iloc[i], outcome[view].iloc[i], eligible.iloc[i]
        d = pd.DataFrame({"change": ch, "share": sh, "outcome": out, "ok": ok})
        d = d[d.ok & d.outcome.notna() & d.change.notna() & d.share.notna()].drop(columns="ok")
        d["date"] = date; d["view"] = view; d["symbol"] = d.index
        rows.append(d)
obs = pd.concat(rows, ignore_index=True)
obs["group"] = obs.symbol.map(GROUPS); obs["year"] = obs.date.dt.year
obs["past_month"] = np.where(obs.change > 0, "up", np.where(obs.change < 0, "down", "flat"))
obs["continued"] = (np.sign(obs.outcome) == np.sign(obs.change)) & (obs.change != 0)

def label(df, hi, lo):
    conf = ((df.change > 0) & (df.share >= hi)) | ((df.change < 0) & (df.share <= lo))
    dont = ((df.change > 0) & (df.share <= lo)) | ((df.change < 0) & (df.share >= hi))
    both = conf & dont                      # only possible when hi == lo and share == hi: treated as Neutral
    return np.where(both, "Neutral", np.where(conf, "Confirms", np.where(dont, "Doesn't confirm", "Neutral")))

obs["label"] = label(obs, 55, 45)
assert obs.date.max() <= pd.Timestamp(config.BUILD_END)
last_outcome_end = P.index[cp[cp + N < len(P)][-1] + N]
print("checkpoints:", len(cp), "| last checkpoint with a full outcome ends", last_outcome_end.date())
print("observations (symbol-checkpoints):", obs.groupby("view").size().to_dict())
obs.to_csv(R / "02_observations.csv", index=False)
obs.head()

checkpoints: 263 | last checkpoint with a full outcome ends 2020-12-14
observations (symbol-checkpoints): {'price': 4939, 'vs sector': 4939}


,change,share,outcome,date,view,symbol,group,year,past_month,continued,label
0,0.035560,48.070930,0.001104,1999-02-03,price,SPY,benchmark (SPY),1999,up,True,Neutral
1,0.135417,57.453661,-0.074228,1999-02-03,price,XLK,sector,1999,up,False,Confirms
2,-0.051574,50.655489,0.033192,1999-02-03,price,XLE,sector,1999,down,False,Neutral
3,0.060844,62.035212,0.023282,1999-02-03,price,XLV,sector,1999,up,True,Confirms
4,-0.018908,44.784116,-0.014454,1999-02-03,price,XLU,sector,1999,down,True,Confirms


## 3. Result 1: continuation rate by label (all checkpoints combined = baseline)

In [4]:
ORDER = ["Confirms", "Doesn't confirm", "Neutral"]
def rate_table(d, by):
    g = d.groupby(by).continued.agg(n="size", continued="sum", rate="mean")
    return g
r1 = []
for view, d in obs.groupby("view"):
    t = rate_table(d, "label").reindex(ORDER)
    base = pd.Series({"n": len(d), "continued": d.continued.sum(), "rate": d.continued.mean()}, name="All checkpoints (baseline)")
    t = pd.concat([t, base.to_frame().T])
    t.insert(0, "view", view); r1.append(t.rename_axis("label").reset_index())
r1 = pd.concat(r1, ignore_index=True)
r1["rate_pct"] = (r1.rate * 100).round(2)
r1.to_csv(R / "02_continuation_by_label.csv", index=False)
print(r1[["view", "label", "n", "continued", "rate_pct"]].to_string(index=False))
for view in ("price", "vs sector"):
    t = r1[r1.view == view].set_index("label")
    c_rate, d_rate = t.loc["Confirms", "rate_pct"], t.loc["Doesn't confirm", "rate_pct"]
    print(f"\n[{view}] Confirms minus Doesn't confirm: {c_rate - d_rate:.2f} points")

     view                      label      n  continued  rate_pct
    price                   Confirms 2915.0     1486.0     50.98
    price            Doesn't confirm  478.0      249.0     52.09
    price                    Neutral 1546.0      805.0     52.07
    price All checkpoints (baseline) 4939.0     2540.0     51.43
vs sector                   Confirms 2879.0     1424.0     49.46
vs sector            Doesn't confirm  476.0      226.0     47.48
vs sector                    Neutral 1584.0      781.0     49.31
vs sector All checkpoints (baseline) 4939.0     2431.0     49.22

[price] Confirms minus Doesn't confirm: -1.11 points

[vs sector] Confirms minus Doesn't confirm: 1.98 points


## 4. Result 2: year by year
Column `confirms_minus_doesnt` = Confirms rate minus Doesn't confirm rate (percentage points). Year = year of the checkpoint date.

In [5]:
yrs = []
for view, d in obs.groupby("view"):
    for y, g in d.groupby("year"):
        row = dict(view=view, year=y, n_all=len(g), rate_all=g.continued.mean() * 100)
        for lab, key in (("Confirms", "confirms"), ("Doesn't confirm", "doesnt"), ("Neutral", "neutral")):
            s = g[g.label == lab]
            row[f"n_{key}"] = len(s); row[f"rate_{key}"] = s.continued.mean() * 100 if len(s) else np.nan
        row["confirms_minus_doesnt"] = row["rate_confirms"] - row["rate_doesnt"]
        yrs.append(row)
yearly = pd.DataFrame(yrs)
yearly.round(1).to_csv(R / "02_by_year.csv", index=False)
for view in ("price", "vs sector"):
    t = yearly[yearly.view == view].set_index("year")
    print(f"=== {view} view ===")
    print(t.drop(columns="view").round(1).to_string())
    pos = (t.confirms_minus_doesnt > 0).sum(); n_ok = t.confirms_minus_doesnt.notna().sum()
    print(f"years where Confirms > Doesn't confirm: {pos} of {len(t)} ({n_ok} years have both labels)\n")

=== price view ===
      n_all  rate_all  n_confirms  rate_confirms  n_doesnt  rate_doesnt  n_neutral  rate_neutral  confirms_minus_doesnt
year                                                                                                                   
1999    118      52.5          63           54.0        18         55.6         37          48.6                   -1.6
2000    143      34.3          78           30.8        26         46.2         39          33.3                  -15.4
2001    156      51.3          93           47.3        27         66.7         36          50.0                  -19.4
2002    160      54.4         106           47.2        15         80.0         39          64.1                  -32.8
2003    175      57.7         101           63.4        22         45.5         52          51.9                   17.9
2004    180      46.7         108           48.1        20         55.0         52          40.4                   -6.9
2005    192      61.5

## 5. Result 3: by group (11 sectors, 4 industries, other markets; SPY shown separately as the benchmark)

In [6]:
grp = []
for (view, gname), d in obs.groupby(["view", "group"]):
    row = dict(view=view, group=gname, n_all=len(d), rate_all=d.continued.mean() * 100)
    for lab, key in (("Confirms", "confirms"), ("Doesn't confirm", "doesnt")):
        s = d[d.label == lab]; row[f"n_{key}"] = len(s); row[f"rate_{key}"] = s.continued.mean() * 100
    row["confirms_minus_doesnt"] = row["rate_confirms"] - row["rate_doesnt"]
    grp.append(row)
grp = pd.DataFrame(grp)
grp.round(1).to_csv(R / "02_by_group.csv", index=False)
print(grp.round(1).to_string(index=False))

     view           group  n_all  rate_all  n_confirms  rate_confirms  n_doesnt  rate_doesnt  confirms_minus_doesnt
    price benchmark (SPY)    262      54.2         155           50.3        13         46.2                    4.2
    price        industry    767      48.4         444           49.3        93         53.8                   -4.4
    price   other markets   1463      51.1         919           50.8       105         47.6                    3.2
    price          sector   2447      52.3        1397           51.7       267         53.6                   -1.9
vs sector benchmark (SPY)    262      50.8         133           51.1        20         50.0                    1.1
vs sector        industry    767      48.1         446           47.1        68         50.0                   -2.9
vs sector   other markets   1463      49.9         898           50.0       101         50.5                   -0.5
vs sector          sector   2447      49.0        1402           49.7   

## 6. Result 4: average next-month change by label, split by whether the past month was up or down (percent)

In [7]:
avg = (obs[obs.past_month != "flat"].groupby(["view", "label", "past_month"])
       .agg(n=("outcome", "size"), mean_next_month_pct=("outcome", lambda s: s.mean() * 100),
            median_next_month_pct=("outcome", lambda s: s.median() * 100), continued_pct=("continued", lambda s: s.mean() * 100))
       .reset_index())
avg["label"] = pd.Categorical(avg.label, ORDER, ordered=True)
avg = avg.sort_values(["view", "past_month", "label"], ascending=[True, False, True])
avg.round(2).to_csv(R / "02_avg_next_month_change.csv", index=False)
print(avg.round(2).to_string(index=False))

     view           label past_month    n  mean_next_month_pct  median_next_month_pct  continued_pct
    price        Confirms         up 1699                 0.44                   0.94          59.33
    price Doesn't confirm         up  308                 0.38                   0.70          59.42
    price         Neutral         up  955                 0.69                   1.24          60.63
    price        Confirms       down 1216                 1.22                   1.41          39.31
    price Doesn't confirm       down  170                 0.34                   1.45          38.82
    price         Neutral       down  579                 1.05                   1.40          39.03
vs sector        Confirms         up 1384                -0.21                  -0.07          48.99
vs sector Doesn't confirm         up  249                -0.18                  -0.14          47.39
vs sector         Neutral         up  828                -0.04                  -0.01      

## 7. Cutoff check (information only; no "best" cutoff is picked)
Same as result 1 with the cutoffs 50/50, 52/48, 55/45, 58/42, 60/40, 65/35. At 50/50 a share of exactly 50 satisfies both rules and is treated as Neutral.

In [8]:
cut = []
for hi in (50, 52, 55, 58, 60, 65):
    lo = 100 - hi
    for view, d in obs.groupby("view"):
        lab = pd.Series(label(d, hi, lo), index=d.index)
        row = dict(view=view, cutoff=f"{hi}/{lo}")
        for name, key in (("Confirms", "confirms"), ("Doesn't confirm", "doesnt"), ("Neutral", "neutral")):
            s = d[lab == name]; row[f"n_{key}"] = len(s); row[f"rate_{key}"] = s.continued.mean() * 100 if len(s) else np.nan
        row["confirms_minus_doesnt"] = row["rate_confirms"] - row["rate_doesnt"]
        cut.append(row)
cut = pd.DataFrame(cut)
cut.round(2).to_csv(R / "02_cutoff_check.csv", index=False)
print(cut.round(2).to_string(index=False))

     view cutoff  n_confirms  rate_confirms  n_doesnt  rate_doesnt  n_neutral  rate_neutral  confirms_minus_doesnt
    price  50/50        3826          50.99      1101        53.50         12          0.00                  -2.50
vs sector  50/50        3820          49.11      1119        49.60          0           NaN                  -0.49
    price  52/48        3495          50.87       816        52.45        628         53.18                  -1.58
vs sector  52/48        3478          48.91       840        49.40        621         50.72                  -0.50
    price  55/45        2915          50.98       478        52.09       1546         52.07                  -1.11
vs sector  55/45        2879          49.46       476        47.48       1584         49.31                   1.98
    price  58/42        2289          50.85       245        55.10       2405         51.60                  -4.25
vs sector  58/42        2251          49.49       281        48.04       2407   

## 8. Extra context (not a prediction): how much could the gap move by chance?
Symbols move together, so observations at one checkpoint are not independent. Here the checkpoints (dates) are resampled as blocks, 2,000 times, to get a rough 95% range for the price-view Confirms minus Doesn't confirm gap. Fixed seed.

In [9]:
rng = np.random.default_rng(0)
d = obs[obs.view == "price"].copy(); d["label"] = label(d, 55, 45)
dates = np.array(sorted(d.date.unique()))
grp_by_date = {k: g for k, g in d.groupby("date")}
agg = pd.DataFrame({dt: {"c_n": ((g.label == "Confirms")).sum(), "c_k": ((g.label == "Confirms") & g.continued).sum(),
                         "d_n": ((g.label == "Doesn't confirm")).sum(), "d_k": ((g.label == "Doesn't confirm") & g.continued).sum()}
                    for dt, g in grp_by_date.items()}).T
gaps = []
for _ in range(2000):
    pick = rng.integers(0, len(agg), len(agg)); s = agg.iloc[pick].sum()
    gaps.append((s.c_k / s.c_n - s.d_k / s.d_n) * 100)
lo, hi = np.percentile(gaps, [2.5, 97.5])
s = agg.sum(); point = (s.c_k / s.c_n - s.d_k / s.d_n) * 100
print(f"price view gap: {point:.2f} points; rough 95% range {lo:.2f} to {hi:.2f} (resampling {len(agg)} checkpoint dates)")

price view gap: -1.11 points; rough 95% range -7.65 to 4.91 (resampling 262 checkpoint dates)


## 9. Checks for the three Part A predictions (price view, the main test)

In [10]:
t = r1[r1.view == "price"].set_index("label")
conf, dont, base = t.loc["Confirms"], t.loc["Doesn't confirm"], t.loc["All checkpoints (baseline)"]
gap = (conf.rate - dont.rate) * 100
print(f"V1: Confirms {conf.rate*100:.2f}% (n={int(conf.n)}) vs Doesn't confirm {dont.rate*100:.2f}% (n={int(dont.n)}): gap {gap:.2f} points (need >= 5) -> {'CONFIRMED' if gap >= 5 else 'CONTRADICTED'}")
y = yearly[yearly.view == "price"]
wins = int((y.confirms_minus_doesnt > 0).sum())
print(f"V2: Confirms > Doesn't confirm in {wins} of {len(y)} years (need >= 15) -> {'CONFIRMED' if wins >= 15 else 'CONTRADICTED'}")
print(f"V3: Doesn't confirm continued {dont.rate*100:.2f}% of the time (need < 50) -> {'CONFIRMED' if dont.rate < 0.5 else 'CONTRADICTED'}")
print(f"baseline (all checkpoints): {base.rate*100:.2f}% continued (n={int(base.n)})")

V1: Confirms 50.98% (n=2915) vs Doesn't confirm 52.09% (n=478): gap -1.11 points (need >= 5) -> CONTRADICTED
V2: Confirms > Doesn't confirm in 11 of 22 years (need >= 15) -> CONTRADICTED
V3: Doesn't confirm continued 52.09% of the time (need < 50) -> CONTRADICTED
baseline (all checkpoints): 51.43% continued (n=4939)


---
# Part B: test years (2021-01-01 to the last date in the data)
Checked **once**. Predictions W1-W3 are in FINDINGS.md (written before this ran). Method is exactly Part A's: same rules, same 55/45 cutoffs, same 21-day non-overlapping checkpoints, same 18-of-21 volume minimum, same 23 symbols (no BTC-USD), same two views.

Choices stated up front:
- The checkpoint grid **continues from Part A** (every 21st trading day counted from 1999-01-04). Test checkpoints are those dated 2021-01-01 or later; their 21-day look-back may use late-2020 prices.
- Only checkpoints whose **full next 21 trading days fall inside the data** are used; the rest are dropped and counted below.
- Safeguard: the Part B code is run on the build years first and must reproduce Part A's observations and tables exactly.

In [11]:
P_START = config.TEST_START
P_END = None            # all data through the last date
OUT = config.RESULTS_DIR
R = OUT; R.mkdir(parents=True, exist_ok=True)                 # Part B files are written with the 02b_ prefix
dfF = load_prices(symbols=SYMS, fields=["adj_close", "volume"], end=P_END)
PF = wide(dfF, "adj_close")[SYMS]; VF = wide(dfF, "volume")[SYMS]
print("loaded:", PF.index.min().date(), "to", PF.index.max().date(), "|", len(PF), "days |", PF.shape[1], "symbols")

loaded: 1999-01-04 to 2026-10-15 | 6989 days | 23 symbols


## B-1. The Part A rule as one function, with a guard
Same calculations as Part A, wrapped so they can be applied to any date range.

In [12]:
def compute_obs(P, V):
    V0 = V.fillna(0)
    valid_hist = P.notna().astype(int).rolling(N + 1, min_periods=N + 1).sum() == N + 1
    nz_days = (V0 > 0).astype(int).rolling(N, min_periods=N).sum()
    eligible = valid_hist & (nz_days >= MIN_VOL_DAYS)
    S = P[config.SECTORS]
    sec_change = (S / S.shift(N) - 1).mean(axis=1, skipna=True)
    sec_outcome = (S.shift(-N) / S - 1).mean(axis=1, skipna=True)
    sec_daily = S.pct_change(fill_method=None).mean(axis=1, skipna=True)
    def up_share_(daily_move):
        up = (daily_move > 0)
        up_vol = V0.where(up, 0.0).rolling(N, min_periods=N).sum()
        tot = V0.rolling(N, min_periods=N).sum()
        return 100 * up_vol / tot.where(tot > 0)
    d_price = P.diff()
    d_rel = P.pct_change(fill_method=None).sub(sec_daily, axis=0)
    chg = {"price": P / P.shift(N) - 1, "vs sector": (P / P.shift(N) - 1).sub(sec_change, axis=0)}
    outc = {"price": P.shift(-N) / P - 1, "vs sector": (P.shift(-N) / P - 1).sub(sec_outcome, axis=0)}
    shr = {"price": up_share_(d_price), "vs sector": up_share_(d_rel)}
    cp_ = np.arange(N, len(P), N)
    rows_ = []
    for view in chg:
        for i in cp_:
            d = pd.DataFrame({"change": chg[view].iloc[i], "share": shr[view].iloc[i], "outcome": outc[view].iloc[i], "ok": eligible.iloc[i]})
            d = d[d.ok & d.outcome.notna() & d.change.notna() & d.share.notna()].drop(columns="ok")
            d["date"] = P.index[i]; d["view"] = view; d["symbol"] = d.index
            rows_.append(d)
    o = pd.concat(rows_, ignore_index=True)
    o["group"] = o.symbol.map(GROUPS); o["year"] = o.date.dt.year
    o["past_month"] = np.where(o.change > 0, "up", np.where(o.change < 0, "down", "flat"))
    o["continued"] = (np.sign(o.outcome) == np.sign(o.change)) & (o.change != 0)
    o["label"] = label(o, 55, 45)
    return o, cp_

def rate_by_label(o):
    out = []
    for view, d in o.groupby("view"):
        t = d.groupby("label").continued.agg(n="size", continued="sum", rate="mean").reindex(ORDER)
        base = pd.Series({"n": len(d), "continued": d.continued.sum(), "rate": d.continued.mean()}, name="All checkpoints (baseline)")
        t = pd.concat([t, base.to_frame().T]); t.insert(0, "view", view)
        out.append(t.rename_axis("label").reset_index())
    out = pd.concat(out, ignore_index=True); out["rate_pct"] = (out.rate * 100).round(2)
    return out

def by_year(o):
    yrs_ = []
    for view, d in o.groupby("view"):
        for y, g in d.groupby("year"):
            row = dict(view=view, year=y, n_all=len(g), rate_all=g.continued.mean() * 100)
            for lab, key in (("Confirms", "confirms"), ("Doesn't confirm", "doesnt"), ("Neutral", "neutral")):
                s = g[g.label == lab]; row[f"n_{key}"] = len(s); row[f"rate_{key}"] = s.continued.mean() * 100 if len(s) else np.nan
            row["confirms_minus_doesnt"] = row["rate_confirms"] - row["rate_doesnt"]
            yrs_.append(row)
    return pd.DataFrame(yrs_)

def by_group(o):
    g_ = []
    for (view, gname), d in o.groupby(["view", "group"]):
        row = dict(view=view, group=gname, n_all=len(d), rate_all=d.continued.mean() * 100)
        for lab, key in (("Confirms", "confirms"), ("Doesn't confirm", "doesnt")):
            s = d[d.label == lab]; row[f"n_{key}"] = len(s); row[f"rate_{key}"] = s.continued.mean() * 100
        row["confirms_minus_doesnt"] = row["rate_confirms"] - row["rate_doesnt"]
        g_.append(row)
    return pd.DataFrame(g_)

def cutoffs(o):
    c_ = []
    for hi in (50, 52, 55, 58, 60, 65):
        lo = 100 - hi
        for view, d in o.groupby("view"):
            lab = pd.Series(label(d, hi, lo), index=d.index)
            row = dict(view=view, cutoff=f"{hi}/{lo}")
            for name, key in (("Confirms", "confirms"), ("Doesn't confirm", "doesnt"), ("Neutral", "neutral")):
                s = d[lab == name]; row[f"n_{key}"] = len(s); row[f"rate_{key}"] = s.continued.mean() * 100 if len(s) else np.nan
            row["confirms_minus_doesnt"] = row["rate_confirms"] - row["rate_doesnt"]
            c_.append(row)
    return pd.DataFrame(c_)

def gap_resampling(o, n_boot=2000):
    rng_ = np.random.default_rng(0)
    d = o[o.view == "price"]
    agg_ = pd.DataFrame({dt: {"c_n": (g.label == "Confirms").sum(), "c_k": ((g.label == "Confirms") & g.continued).sum(),
                              "d_n": (g.label == "Doesn't confirm").sum(), "d_k": ((g.label == "Doesn't confirm") & g.continued).sum()}
                         for dt, g in d.groupby("date")}).T
    gaps_ = []
    for _ in range(n_boot):
        pick = rng_.integers(0, len(agg_), len(agg_)); s = agg_.iloc[pick].sum()
        gaps_.append((s.c_k / s.c_n - s.d_k / s.d_n) * 100)
    s = agg_.sum()
    return dict(point_gap=(s.c_k / s.c_n - s.d_k / s.d_n) * 100, lo95=np.percentile(gaps_, 2.5), hi95=np.percentile(gaps_, 97.5), checkpoint_dates=len(agg_))

# ---- guard: same code on the build years must reproduce Part A exactly ----
chk, _ = compute_obs(P, V)
pd.testing.assert_frame_equal(chk.reset_index(drop=True), obs.reset_index(drop=True))
pd.testing.assert_frame_equal(rate_by_label(obs), r1)
pd.testing.assert_frame_equal(by_year(obs).reset_index(drop=True), yearly.reset_index(drop=True))
pd.testing.assert_frame_equal(by_group(obs).reset_index(drop=True), grp.reset_index(drop=True))
pd.testing.assert_frame_equal(cutoffs(obs).reset_index(drop=True), cut.reset_index(drop=True))
assert abs(gap_resampling(obs)["point_gap"] - point) < 1e-9
print("guard passed: Part B code reproduces Part A observations, label table, yearly, group, cutoff and resampling results exactly")

guard passed: Part B code reproduces Part A observations, label table, yearly, group, cutoff and resampling results exactly


## B-2. Test-period observations
Checkpoint grid continues from Part A; only checkpoints dated on/after the test start with a full next-21-day outcome inside the data are used.

In [13]:
obsF, cpF = compute_obs(PF, VF)
cp_dates = PF.index[cpF]
test_pos = [i for i in cpF if PF.index[i] >= pd.Timestamp(P_START)]
full_pos = [i for i in test_pos if i + N <= len(PF) - 1]
dropped_pos = [i for i in test_pos if i + N > len(PF) - 1]
print(f"checkpoints dated {pd.Timestamp(P_START).date()} or later: {len(test_pos)} | with full next-21-day outcome: {len(full_pos)} | dropped: {len(dropped_pos)}",
      [PF.index[i].date().isoformat() for i in dropped_pos])
print("first test checkpoint:", PF.index[full_pos[0]].date(), "| last used checkpoint:", PF.index[full_pos[-1]].date(),
      "| its outcome ends", PF.index[full_pos[-1] + N].date())
obsT = obsF[obsF.date >= pd.Timestamp(P_START)].reset_index(drop=True)
print("labeled symbol-checkpoints:", obsT.groupby("view").size().to_dict(), "| symbols per checkpoint: min", obsT[obsT.view == "price"].groupby("date").size().min(),
      "max", obsT[obsT.view == "price"].groupby("date").size().max())
obsT.to_csv(R / "02b_observations.csv", index=False)
obsT.head()

checkpoints dated 2021-01-01 or later: 69 | with full next-21-day outcome: 68 | dropped: 1 ['2026-09-23']
first test checkpoint: 2021-01-14 | last used checkpoint: 2026-08-24 | its outcome ends 2026-09-23
labeled symbol-checkpoints: {'price': 1564, 'vs sector': 1564} | symbols per checkpoint: min 23 max 23


,change,share,outcome,date,view,symbol,group,year,past_month,continued,label
0,0.042267,60.579842,0.036569,2021-01-14,price,SPY,benchmark (SPY),2021,up,True,Confirms
1,0.034387,49.526104,0.073016,2021-01-14,price,XLK,sector,2021,up,True,Neutral
2,0.135960,55.577593,0.029451,2021-01-14,price,XLE,sector,2021,up,True,Confirms
3,0.056395,64.098395,-0.004202,2021-01-14,price,XLV,sector,2021,up,False,Confirms
4,0.015943,41.938290,-0.007374,2021-01-14,price,XLU,sector,2021,up,False,Doesn't confirm


## B-3. Result 1: continuation rate by label, plus baseline (both views)

In [14]:
r1T = rate_by_label(obsT)
r1T.to_csv(R / "02b_continuation_by_label.csv", index=False)
print(r1T[["view", "label", "n", "continued", "rate_pct"]].to_string(index=False))
for view in ("price", "vs sector"):
    t = r1T[r1T.view == view].set_index("label")
    c_rate, d_rate = t.loc["Confirms", "rate_pct"], t.loc["Doesn't confirm", "rate_pct"]
    print(f"[{view}] Confirms minus Doesn't confirm: {c_rate - d_rate:.2f} points")

     view                      label      n  continued  rate_pct
    price                   Confirms  965.0      460.0     47.67
    price            Doesn't confirm   88.0       45.0     51.14
    price                    Neutral  511.0      254.0     49.71
    price All checkpoints (baseline) 1564.0      759.0     48.53
vs sector                   Confirms  914.0      464.0     50.77
vs sector            Doesn't confirm  127.0       47.0     37.01
vs sector                    Neutral  523.0      257.0     49.14
vs sector All checkpoints (baseline) 1564.0      768.0     49.10
[price] Confirms minus Doesn't confirm: -3.47 points
[vs sector] Confirms minus Doesn't confirm: 13.76 points


## B-4. Result 2: year by year (both views)

In [15]:
yearlyT = by_year(obsT)
yearlyT.round(1).to_csv(R / "02b_by_year.csv", index=False)
for view in ("price", "vs sector"):
    t = yearlyT[yearlyT.view == view].set_index("year")
    print(f"=== {view} view ===")
    print(t.drop(columns="view").round(1).to_string())
    print(f"years where Confirms > Doesn't confirm: {(t.confirms_minus_doesnt > 0).sum()} of {len(t)}\n")

=== price view ===
      n_all  rate_all  n_confirms  rate_confirms  n_doesnt  rate_doesnt  n_neutral  rate_neutral  confirms_minus_doesnt
year                                                                                                                   
2021    276      48.9         157           43.9        18         72.2        101          52.5                  -28.3
2022    276      53.6         184           53.3        15         73.3         77          50.6                  -20.1
2023    276      45.7         168           38.1        14         42.9         94          59.6                   -4.8
2024    276      46.0         163           47.2        17         41.2         96          44.8                    6.1
2025    276      50.4         178           54.5        15         40.0         83          43.4                   14.5
2026    184      45.7         115           47.8         9         22.2         60          45.0                   25.6
years where Confirms 

## B-5. Result 3: by group

In [16]:
grpT = by_group(obsT)
grpT.round(1).to_csv(R / "02b_by_group.csv", index=False)
print(grpT.round(1).to_string(index=False))

     view           group  n_all  rate_all  n_confirms  rate_confirms  n_doesnt  rate_doesnt  confirms_minus_doesnt
    price benchmark (SPY)     68      58.8          41           56.1         6         66.7                  -10.6
    price        industry    272      48.9         166           48.8        17         58.8                  -10.0
    price   other markets    476      45.6         293           46.8        28         53.6                   -6.8
    price          sector    748      49.3         465           47.1        37         43.2                    3.9
vs sector benchmark (SPY)     68      54.4          44           63.6         6          0.0                   63.6
vs sector        industry    272      52.2         155           49.0        24         45.8                    3.2
vs sector   other markets    476      50.8         275           51.6        33         48.5                    3.2
vs sector          sector    748      46.4         440           49.5   

## B-6. Result 4 (context, not a prediction): resampling range for the price-view gap
Checkpoint dates resampled as blocks, 2,000 times, fixed seed (same as Part A).

In [17]:
gr = gap_resampling(obsT)
pd.DataFrame([gr]).round(2).to_csv(R / "02b_gap_resampling.csv", index=False)
print(f"price view gap: {gr['point_gap']:.2f} points; rough 95% range {gr['lo95']:.2f} to {gr['hi95']:.2f} (resampling {gr['checkpoint_dates']} checkpoint dates)")

price view gap: -3.47 points; rough 95% range -16.36 to 10.00 (resampling 68 checkpoint dates)


## B-7. Result 5: cutoff check (information only; no best cutoff is picked)

In [18]:
cutT = cutoffs(obsT)
cutT.round(2).to_csv(R / "02b_cutoff_check.csv", index=False)
print(cutT.round(2).to_string(index=False))

     view cutoff  n_confirms  rate_confirms  n_doesnt  rate_doesnt  n_neutral  rate_neutral  confirms_minus_doesnt
    price  50/50        1279          48.24       285        49.82          0           NaN                  -1.58
vs sector  50/50        1224          50.41       340        44.41          0           NaN                   6.00
    price  52/48        1159          48.23       186        50.00        219         48.86                  -1.77
vs sector  52/48        1111          50.77       233        37.77        220         52.73                  13.00
    price  55/45         965          47.67        88        51.14        511         49.71                  -3.47
vs sector  55/45         914          50.77       127        37.01        523         49.14                  13.76
    price  58/42         743          48.59        34        47.06        787         48.54                   1.53
vs sector  58/42         703          50.07        62        32.26        799   

## B-8. Result 6: build vs test, gap and baseline (both views)

In [19]:
rows = []
for view in ("price", "vs sector"):
    a = r1[r1.view == view].set_index("label"); b = r1T[r1T.view == view].set_index("label")
    row = dict(view=view)
    for tag, t in (("build", a), ("test", b)):
        row[f"{tag}_confirms_pct"] = t.loc["Confirms", "rate_pct"]; row[f"{tag}_doesnt_pct"] = t.loc["Doesn't confirm", "rate_pct"]
        row[f"{tag}_gap"] = t.loc["Confirms", "rate_pct"] - t.loc["Doesn't confirm", "rate_pct"]
        row[f"{tag}_baseline_pct"] = t.loc["All checkpoints (baseline)", "rate_pct"]
        row[f"{tag}_n_confirms"] = int(t.loc["Confirms", "n"]); row[f"{tag}_n_doesnt"] = int(t.loc["Doesn't confirm", "n"])
        row[f"{tag}_n_all"] = int(t.loc["All checkpoints (baseline)", "n"])
    rows.append(row)
bt = pd.DataFrame(rows).round(2)
bt.to_csv(R / "02b_build_vs_test.csv", index=False)
bt.T

,0,1
view,price,vs sector
build_confirms_pct,50.98,49.46
build_doesnt_pct,52.09,47.48
build_gap,-1.11,1.98
build_baseline_pct,51.43,49.22
build_n_confirms,2915,2879
build_n_doesnt,478,476
build_n_all,4939,4939
test_confirms_pct,47.67,50.77
test_doesnt_pct,51.14,37.01


## B-9. Checks for the three Part B predictions

In [20]:
tp = r1T[r1T.view == "price"].set_index("label"); ts = r1T[r1T.view == "vs sector"].set_index("label")
gp = tp.loc["Confirms", "rate_pct"] - tp.loc["Doesn't confirm", "rate_pct"]
gs = ts.loc["Confirms", "rate_pct"] - ts.loc["Doesn't confirm", "rate_pct"]
bp, bs = tp.loc["All checkpoints (baseline)", "rate_pct"], ts.loc["All checkpoints (baseline)", "rate_pct"]
print(f"W1 price view gap: {gp:.2f} points (need between -5 and +5) -> {'CONFIRMED' if -5 <= gp <= 5 else 'CONTRADICTED'}")
print(f"W2 vs-sector gap:  {gs:.2f} points (need between -5 and +5) -> {'CONFIRMED' if -5 <= gs <= 5 else 'CONTRADICTED'}")
print(f"W3 price-view baseline: {bp:.2f}% (need above 50) -> {'CONFIRMED' if bp > 50 else 'CONTRADICTED'}   [vs-sector baseline for reference: {bs:.2f}%]")

W1 price view gap: -3.47 points (need between -5 and +5) -> CONFIRMED
W2 vs-sector gap:  13.76 points (need between -5 and +5) -> CONTRADICTED
W3 price-view baseline: 48.53% (need above 50) -> CONTRADICTED   [vs-sector baseline for reference: 49.10%]
